# 06 · Suffix leakage: the apostrophe gives the name away

**Katip lab, notebook 6.** About 2 minutes. No training.

### The problem

Turkish writes case suffixes on names after an apostrophe, and the suffix **agrees with the name**:
*Ahmet'in*, *Ayşe'nin*, *Öztürk'ün*, *Kaya'ya*, *Demir'e*, *Yılmaz'da*, *Ahmet'te*. Katip masks the
name and keeps the suffix:

> Davacı **[AD SOYAD]'ün** vekili …

The *'ün* says: the hidden surname's last vowel is *ö* or *ü*, and it ends in a consonant. Vowel
harmony, buffer letters (*n*, *y*) and *d/t* assimilation together leak part of the name. Every
masked name with a suffix is a small clue; a document mentions the same person many times, in
different cases, and the clues add up.

### Questions

1. How common are suffixed names? (If rare, we stop here.)
2. How much does one suffix reveal, in bits, and how small does it make the candidate set?
3. Does a simple fix (re-inflecting the suffix to agree with the placeholder) remove the leak while
   keeping the sentence grammatical?

In [1]:
import math, re, statistics
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import katiplab as K
import suffix as S

sets = {"Katip synthetic (blind + heldout)": K.load_synth("blind") + K.load_synth("heldout"),
        "Katip synthetic training data": K.load_synth("train"),
        "real Turkish text (Wiki NER dev + test)": K.load_wiki("dev") + K.load_wiki("test")}

def mentions(docs):
    """(doc id, name without suffix, suffix or None) for every gold name."""
    for d in docs:
        t = d["text"]
        for a, b, typ in d["spans"]:
            if typ != "PERSON": continue
            name = t[a:b]; m = re.search(r"['’]([a-zçğıöşü]+)$", name)
            yield d["id"], (name[:m.start()] if m else name), (m.group(1) if m else S.suffix_after(t, b))

## 6.1 · How common?

In [2]:
rows = []
for k, docs in sets.items():
    ms = list(mentions(docs)); n = sum(s is not None for *_, s in ms)
    rows.append({"set": k, "names": len(ms), "with apostrophe suffix": n, "share": n / len(ms)})
pd.DataFrame(rows).style.format({"share": "{:.0%}"})

,set,names,with apostrophe suffix,share
0,Katip synthetic (blind + heldout),640,25,4%
1,Katip synthetic training data,36663,1332,4%
2,real Turkish text (Wiki NER dev + test),1295,366,28%


**Read this carefully.** In real prose more than a quarter of names carry a suffix; Katip's
synthetic documents have very few, so our benchmark has never measured this. Real petitions are full
of *davacı X'in*, *davalı Y'ye*, *tanık Z'nin*. The issue is common enough to fix.

## 6.2 · How much does one suffix reveal?

The candidate pool is every first name and surname in Katip's Turkish name lists. For each observed
suffix we ask the module which candidates would produce *exactly* that suffix
(`S.compatible`), and convert the shrink into bits: log₂(pool / compatible). With a uniform prior this
is the information the suffix gives an attacker who knows only that a Turkish name was hidden.

In [3]:
b = K._bundle / "katip_redact/data"
POOL = sorted({w.title() for f in ("first_names.txt", "last_names.txt") for w in (b / f).read_text(encoding="utf-8").split()})
print(len(POOL), "candidate names")
for name, suf in [("Ahmet", "in"), ("Ayşe", "nin"), ("Öztürk", "ün"), ("Arzu", "nun"), ("Yılmaz", "da"), ("Ahmet", "te")]:
    c = S.compatible(suf, POOL)
    print(f"  [***]'{suf:4} -> {len(c):5} candidates ({S.leak_bits(suf, POOL):.1f} bits)   e.g. {', '.join(c[:5])}")

2852 candidate names
  [***]'in   ->   884 candidates (1.7 bits)   e.g. Abdiş, Abdulaziz, Abdulbekir, Abdulkadir, Abdulkerim
  [***]'nin  ->   394 candidates (2.9 bits)   e.g. Abdulgazi, Abdülhadi, Abide, Abiye, Adviye
  [***]'ün   ->   105 candidates (4.8 bits)   e.g. Adıgün, Akgöl, Akgül, Akgündüz, Aksöğüt
  [***]'nun  ->    62 candidates (5.5 bits)   e.g. Aksu, Akçasu, Alexandru, Argu, Aru
  [***]'da   ->  1156 candidates (1.3 bits)   e.g. Abdul, Abdulselam, Abdurrahman, Abdülcemal, Abuzar
  [***]'te   ->   268 candidates (3.4 bits)   e.g. Abdiş, Abdullatif, Abdulmelik, Abdulsamet, Abdulsemet


In [4]:
ms = [m for m in mentions(sets["real Turkish text (Wiki NER dev + test)"]) if m[2]]
bits = [S.leak_bits(s, POOL) for *_, s in ms]
by = Counter(s for *_, s in ms)
tab = pd.DataFrame([{"suffix": "'" + s, "seen": n, "candidates left": len(S.compatible(s, POOL)), "bits": S.leak_bits(s, POOL)}
                    for s, n in by.most_common(14)])
print(f"real text: {len(ms)} suffixed names, mean {statistics.mean(bits):.2f} bits each "
      f"(candidate set shrinks {2 ** statistics.mean(bits):.1f}x on average)")
tab.style.format({"bits": "{:.2f}"})

real text: 366 suffixed names, mean 2.44 bits each (candidate set shrinks 5.4x on average)


,suffix,seen,candidates left,bits
0,'ın,75,901,1.66
1,'in,62,884,1.69
2,'nin,36,394,2.86
3,'un,29,228,3.64
4,'nın,23,251,3.51
5,'nun,21,62,5.52
6,'a,17,1129,1.34
7,'i,16,884,1.69
8,'e,15,989,1.53
9,'yı,8,251,3.51


## 6.3 · Clues add up within a document

A case file mentions the same person many times, in different cases: *X'in*, *X'e*, *X'de*,
*X'le*. Each form tells a little; together they tell more. The cell below takes every name in the
pool, writes down the suffix it would take in each case (its **signature**), and groups names with the
same signature. The group size is how many candidates an attacker is left with after seeing those
forms.

In [5]:
CASES = {"genitive 'nin": "nin", "dative 'ye": "ye", "locative 'de": "de", "instrumental 'le": "le"}
def signature(name, cases): return tuple(S.reinflect(CASES[c], name) for c in cases)
rows = []
for k in range(1, len(CASES) + 1):
    cases = list(CASES)[:k]
    groups = Counter(signature(n, cases) for n in POOL)
    sizes = [groups[signature(n, cases)] for n in POOL]
    rows.append({"forms seen": " + ".join(c.split()[1] for c in cases), "distinct signatures": len(groups),
                 "median candidates left": statistics.median(sizes), "smallest group": min(sizes),
                 "mean bits": statistics.mean(math.log2(len(POOL) / z) for z in sizes)})
pd.DataFrame(rows).style.format({"mean bits": "{:.2f}"})

,forms seen,distinct signatures,median candidates left,smallest group,mean bits
0,'nin,8,884.000000,27,2.40
1,'nin + 'ye,8,884.000000,27,2.40
2,'nin + 'ye + 'de,12,394.000000,21,3.02
3,'nin + 'ye + 'de + 'le,12,394.000000,21,3.02


## 6.4 · The fix: re-inflect the suffix to agree with the placeholder

Three ways to write a masked name with a suffix:

| mode | output | leaks? | grammatical? |
| --- | --- | --- | --- |
| keep (today) | `[AD SOYAD]'ün vekili` | yes | no: *SOYAD'ün* is wrong Turkish |
| drop | `[AD SOYAD] vekili` | no | loses the case; an LLM reading it loses who did what to whom |
| **re-inflect** | `[AD SOYAD]'ın vekili` | **no** | **yes**: the suffix now agrees with *SOYAD* |

Re-inflection keeps the case (genitive stays genitive) and drops everything the suffix knew about the
name. For a black-box PDF the equivalent is to extend the box over the suffix.

In [6]:
text = "Davalı Gönül Öztürk'ün vekili, davacı Arzu Kaya'ya ve tanık Ahmet Yılmaz'a ait beyanları Ahmet'te bulunan dosyaya ekledi."
spans = [(m.start(), m.end()) for m in re.finditer(r"Gönül Öztürk|Arzu Kaya|Ahmet Yılmaz|Ahmet(?=')", text)]
for mode in ("keep", "drop", "reinflect"):
    print(f"{mode:10}", S.apply_mask(text, spans, "[AD SOYAD]", mode))

keep       Davalı [AD SOYAD]'ün vekili, davacı [AD SOYAD]'ya ve tanık [AD SOYAD]'a ait beyanları [AD SOYAD]'te bulunan dosyaya ekledi.
drop       Davalı [AD SOYAD] vekili, davacı [AD SOYAD] ve tanık [AD SOYAD] ait beyanları [AD SOYAD] bulunan dosyaya ekledi.
reinflect  Davalı [AD SOYAD]'ın vekili, davacı [AD SOYAD]'a ve tanık [AD SOYAD]'a ait beyanları [AD SOYAD]'da bulunan dosyaya ekledi.


In [7]:
leak_before, leak_after, changed = [], [], 0
for *_, s in ms:
    new = S.reinflect(s, "SOYAD")
    leak_before.append(S.leak_bits(s, POOL))
    leak_after.append(0.0)                       # the new suffix depends only on the placeholder
    changed += new != s
print(f"suffixed names in real text: {len(ms)}")
print(f"mean leak, keep:      {statistics.mean(leak_before):.2f} bits per name")
print(f"mean leak, reinflect: {statistics.mean(leak_after):.2f} bits per name  ({changed} of {len(ms)} suffixes rewritten)")
assert all(S.reinflect(S.reinflect(s, 'SOYAD'), 'SOYAD') == S.reinflect(s, 'SOYAD') for *_, s in ms), "not stable"

suffixed names in real text: 366
mean leak, keep:      2.44 bits per name
mean leak, reinflect: 0.00 bits per name  (254 of 366 suffixes rewritten)


## 6.5 · Sanity check: the module knows Turkish

The leak numbers are only as good as the suffix rules. Check them against names whose correct
suffixes we know, and look at any failure before believing 6.2.

In [8]:
GOLD = [("Ahmet", "in", "e", "te"), ("Ayşe", "nin", "ye", "de"), ("Öztürk", "ün", "e", "te"), ("Kaya", "nın", "ya", "da"),
        ("Arzu", "nun", "ya", "da"), ("Gül", "ün", "e", "de"), ("Yılmaz", "ın", "a", "da"), ("Ali", "nin", "ye", "de"),
        ("Bengü", "nün", "ye", "de"), ("Kurt", "un", "a", "ta"), ("Özdemir", "in", "e", "de"), ("Aslı", "nın", "ya", "da")]
bad = []
for name, gen, dat, loc in GOLD:
    for template, want in (("nin", gen), ("ye", dat), ("de", loc)):
        got = S.reinflect(template, name)
        if got != want: bad.append((name, template, got, want))
print("errors:", bad or "none")
assert not bad

errors: none


## Conclusion you should be able to state

> About a quarter of names in real Turkish prose carry an apostrophe suffix, and Katip keeps it after
> the mask. One suffix reveals about the measured number of bits about the hidden name (up to ~7 bits
> for rare forms like *'nün*), and repeated mentions in one document add up. Re-inflecting the suffix
> to agree with the placeholder removes the leak (0 bits) and makes the masked text grammatical, which
> also helps an LLM that reads it through the Kalkan gateway.

This is a 30-line change to the masking step, not a model change. To ship it, call
`suffix.reinflect` in label mode and extend the box over the suffix in black-box mode.